<a href="https://colab.research.google.com/github/HiveCase/DLP-DA5013/blob/main/Week3/GPT_2_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer, DataCollatorForLanguageModeling, GPT2Config, GPT2LMHeadModel, TrainingArguments, Trainer
import wandb
import pandas as pd
from pprint import pprint
import json
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# !pip install -U "datasets==3.6.0"

In [3]:
ds = load_dataset('bookcorpus/bookcorpus', split='all')
pprint(ds)

The repository for bookcorpus/bookcorpus contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/bookcorpus/bookcorpus.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split:   0%|          | 0/74004228 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 74004228
})


In [4]:
# Number of words in Bookcorpus dataset
total_words = sum(
    len(example["text"].split())
    for example in ds
)
print(f"The number of words in Bookcorpus Dataset is: {total_words}")

The number of words in Bookcorpus Dataset is: 984845743


In [8]:
tokenizer = AutoTokenizer.from_pretrained('gpt2')
print(tokenizer)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

GPT2Tokenizer(name_or_path='gpt2', vocab_size=50257, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
})


In [9]:
bs = 4
model_inputs = tokenizer(ds[0:bs]['text'], paddings=True)
pprint(model_inputs['input_ids'], compact=True)

[[23073, 837, 339, 561, 307, 24447, 1088, 262, 2877, 2119, 837, 2712, 351, 465,
  14958, 764],
 [4360, 655, 530, 804, 379, 257, 28365, 1908, 683, 14547, 3797, 265, 9229, 764],
 [5562, 550, 587, 502, 1030, 705, 82, 1410, 618, 673, 1392, 683, 12049, 2961,
  764],
 [258, 705, 67, 1775, 262, 3807, 2048, 416, 7457, 837, 6402, 339, 373, 257,
  1310, 1862, 329, 262, 23241, 16251, 837, 475, 351, 4697, 28706, 837, 1863,
  351, 607, 9397, 837, 285, 888, 373, 1690, 7362, 284, 1243, 326, 547, 4697,
  764]]


In [10]:
pprint({id: example for id, example in enumerate(ds[100:112]['text'])})

{0: 'he loved sitting between her brothers on the couch and watching tv .',
 1: 'while it was good that he had so many male role models , she only hoped '
    "he had n't inherited too much of his father 's personality .",
 2: 'after megan watched them disappear in the crowd of family and friends '
    'waiting in the church alcove , she bypassed everyone by turning right and '
    'heading down the hallway .',
 3: 'at the last door on the right , she knocked .',
 4: "`` it 's me , megan . ''",
 5: "emma 's best friend , casey , answered the door .",
 6: "`` well , if it is n't the fairy godmother , '' she mused with a grin .",
 7: 'after megan stepped inside , casey threw her arms around her .',
 8: "megan had only met her a few times , but it was hard not liking emma 's "
    'vivacious and outgoing friend .',
 9: "casey 's long brown hair was pulled back in a lose knot , and she wore a "
    'demure black slip dress and heels .',
 10: "`` so how 's it going ? ''",
 11: "megan asked 

In [11]:
tokenizer.pad_token = '<|endoftext|>'

In [12]:
ds_split = ds.train_test_split(test_size=0.007, seed=42)
pprint(ds_split)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 73486198
    })
    test: Dataset({
        features: ['text'],
        num_rows: 518030
    })
})


In [13]:
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

In [ ]:
from torch.utils.data import DataLoader
from transformers import DataCollatorForLanguageModeling

# Tokenize the text column
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
    )

tokenized_train = ds_split["train"].map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
)

# Create the collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

# Create the DataLoader using tokenized data
dataloader = DataLoader(
    dataset=tokenized_train,
    collate_fn=data_collator,
    batch_size=4,
)

# Inspect the first batch
for batch in dataloader:
    pprint(batch)
    break

Map:   0%|          | 0/73486198 [00:00<?, ? examples/s]

In [ ]:
config = GPT2config()
pprint(config)

In [ ]:
model = GPT2LMHeadModel(config)
pprint(model)